In [ ]:
#!/usr/bin/env python3
"""
Plot ELECT and VDW interaction-energy components across systems.

Example:
    python plot_interaction_energy_components.py \
        --input k_interaction_energy.npz \
        --output k_interaction_energy_components.png \
        --ion K \
        --error-bars
"""

import argparse
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


SYSTEM_LABELS = {
    "mmgbsa310": "3.10",
    "mmgbsa360": "3.60",
    "mmgbsa412": "4.12",
    "mmgbsa436": "4.36",
    "mmgbsa564": "5.64",
    "mmgbsa606": "6.06",
    "mmgbsa687": "6.87",
    "mmgbsa688": "6.88",
    "mmgbsa688_2": "6.88_2",
    "mmgbsa721": "7.21",
}

KEY_PATTERN = re.compile(
    r"^(?P<system>.+)_rep(?P<replica>\d+)_Delta_"
    r"(?P<term>ELECT|VDW)$"
)


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Plot interaction-energy components."
    )

    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Input NPZ file.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output figure file.",
    )

    parser.add_argument(
        "--ion",
        choices=["K", "Na"],
        required=True,
        help="Ion environment displayed in the title.",
    )

    parser.add_argument(
        "--error-bars",
        action="store_true",
        help="Show standard deviations across replicas.",
    )

    parser.add_argument(
        "--dpi",
        type=int,
        default=300,
        help="Output resolution. Default: 300.",
    )

    parser.add_argument(
        "--no-show",
        action="store_true",
        help="Do not display the figure.",
    )

    return parser.parse_args()


def parse_key(key):
    """解析 NPZ key 中的系統、replica 與能量項。"""

    match = KEY_PATTERN.fullmatch(key)

    if match is None:
        return None

    return {
        "system": match.group("system"),
        "replica": int(match.group("replica")),
        "term": match.group("term"),
    }


def summarize_replicas(data):
    """
    先計算每個 replica 的時間平均，再計算跨 replica 統計。

    每個 replica 具有相同權重，不會因 frame 數較多而取得較高權重。
    """

    replica_means = {}

    for key in data.files:
        key_info = parse_key(key)

        if key_info is None:
            print(f"[警告] 無法解析 key，跳過：{key}")
            continue

        system = key_info["system"]
        replica = key_info["replica"]
        term = key_info["term"]

        values = np.asarray(data[key], dtype=float)

        if values.size == 0:
            print(f"[警告] 空資料，跳過：{key}")
            continue

        replica_means.setdefault(
            system,
            {"ELECT": {}, "VDW": {}},
        )

        replica_means[system][term][replica] = np.mean(values)

    summary = {}

    for system, terms in replica_means.items():
        if not terms["ELECT"] or not terms["VDW"]:
            print(f"[警告] {system} 缺少 ELECT 或 VDW，跳過。")
            continue

        summary[system] = {}

        for term in ("ELECT", "VDW"):
            values = np.asarray(
                list(terms[term].values()),
                dtype=float,
            )

            summary[system][term] = {
                "mean": np.mean(values),
                "std": (
                    np.std(values, ddof=1)
                    if len(values) > 1
                    else 0.0
                ),
                "replicas": len(values),
            }

    return summary


def system_sort_key(system):
    """依 docking score 排序，並將 6.88_2 放在 6.88 後方。"""

    label = SYSTEM_LABELS.get(system, system)

    match = re.search(r"\d+\.\d+", label)

    if match:
        score = float(match.group())
    else:
        score = float("inf")

    pose_order = 2 if "_2" in label else 1

    return score, pose_order


def get_display_label(system):
    """將系統名稱轉換為適合圖表顯示的 docking score。"""

    score = SYSTEM_LABELS.get(system)

    if score is None:
        return system

    return f"Docking score={score}"


def plot_summary(summary, args):
    """繪製 ELECT 與 VDW 分組長條圖。"""

    systems = sorted(
        summary,
        key=system_sort_key,
    )

    if not systems:
        raise RuntimeError("沒有有效系統可以繪製。")

    elect_means = np.asarray([
        summary[system]["ELECT"]["mean"]
        for system in systems
    ])

    vdw_means = np.asarray([
        summary[system]["VDW"]["mean"]
        for system in systems
    ])

    elect_std = np.asarray([
        summary[system]["ELECT"]["std"]
        for system in systems
    ])

    vdw_std = np.asarray([
        summary[system]["VDW"]["std"]
        for system in systems
    ])

    labels = [
        get_display_label(system)
        for system in systems
    ]

    x_positions = np.arange(len(systems))
    bar_width = 0.36

    fig, ax = plt.subplots(
        figsize=(15, 10),
        dpi=150,
    )

    elect_yerr = elect_std if args.error_bars else None
    vdw_yerr = vdw_std if args.error_bars else None

    ax.bar(
        x_positions - bar_width / 2,
        elect_means,
        bar_width,
        yerr=elect_yerr,
        capsize=6 if args.error_bars else 0,
        label="Electrostatic",
        color="#4DBBD5",
        alpha=0.9,
    )

    ax.bar(
        x_positions + bar_width / 2,
        vdw_means,
        bar_width,
        yerr=vdw_yerr,
        capsize=6 if args.error_bars else 0,
        label="van der Waals",
        color="#00A087",
        alpha=0.9,
    )

    ax.set_title(
        f"Interaction Energy Components ({args.ion}⁺)",
        fontsize=32,
        fontweight="bold",
        pad=20,
    )

    ax.set_ylabel(
        "Interaction Energy (kcal/mol)",
        fontsize=30,
        fontweight="bold",
        labelpad=15,
    )

    ax.set_xlabel(
        "Initial Docking Pose",
        fontsize=30,
        fontweight="bold",
        labelpad=15,
    )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        labels,
        rotation=20,
        ha="right",
        fontsize=20,
    )

    ax.tick_params(
        axis="y",
        which="major",
        labelsize=24,
    )

    ax.set_xlim(
        -0.5,
        len(systems) - 0.5,
    )

    ax.axhline(
        0,
        color="black",
        linewidth=1.2,
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4,
    )

    ax.legend(
        fontsize=20,
        loc="upper right",
        frameon=True,
    )

    plt.tight_layout()

    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.savefig(
        args.output,
        dpi=args.dpi,
        bbox_inches="tight",
    )

    print(f"圖片已儲存：{args.output}")

    if args.no_show:
        plt.close(fig)
    else:
        plt.show()


def main():
    """主程式。"""

    args = parse_arguments()

    if not args.input.exists():
        raise FileNotFoundError(
            f"找不到輸入檔案：{args.input}"
        )

    with np.load(
        args.input,
        allow_pickle=False,
    ) as data:
        summary = summarize_replicas(data)

    print("\n跨 replica 統計：")

    for system in sorted(summary, key=system_sort_key):
        elect = summary[system]["ELECT"]
        vdw = summary[system]["VDW"]

        print(
            f"{system}: "
            f"ELECT={elect['mean']:.3f} ± {elect['std']:.3f}, "
            f"VDW={vdw['mean']:.3f} ± {vdw['std']:.3f}, "
            f"n={elect['replicas']}"
        )

    plot_summary(summary, args)


if __name__ == "__main__":
    main()